In [ ]:
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from pathlib import Path 
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import matplotlib.pyplot as plt
import numpy as np
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import re
from openai import OpenAI
from bertopic.representation import OpenAI as OpenAIRepresentation
from dotenv import load_dotenv 
import os
import time
load_dotenv()

In [ ]:
DIRECTORY = Path.cwd().parent / "csv"
macro_groups_df= pd.read_csv(DIRECTORY / "macro_community_groups.csv")
macro_groups_df.head(5)

In [ ]:
def clean_text(text):
    text = text.strip()
    text = re.sub(r'^abstract\s*[:.\-\u2014\u2013]?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'doi:\S+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\d+[\d,;\s\-]*\]', '', text)
    text = re.sub(r'\{[^}]*\}', '', text)
    text = re.sub(r'[\[\](){}]', '', text)
    text = re.sub(r'[\u2010\u2011\u2012\u2013\u2014\u2015]', '-', text)
    text = re.sub(r'[^\x20-\x7E\-]', ' ', text)
    text = re.sub(r'[@#$%^&*_=+|\\<>~`/\"\':;!?,]', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

def clean_joined_field(field_value):
    items = str(field_value).split('; ')
    cleaned = []
    for item in items:
        c = clean_text(item)
        if c:
            cleaned.append(c)
    return '; '.join(cleaned)


macro_groups_df['PaperTitles'] = macro_groups_df['PaperTitles'].apply(
    lambda x: clean_joined_field(x) if pd.notna(x) else x
)

macro_groups_df['Abstracts'] = macro_groups_df['Abstracts'].apply(
    lambda x: clean_joined_field(x) if pd.notna(x) else x
)

macro_groups_df.head(5)


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
macro_community_topics = {}

representation_model = MaximalMarginalRelevance(diversity=0.3)

academic_stop_words = list(ENGLISH_STOP_WORDS) + [
    'paper', 'propose', 'proposed', 'based', 'using', 'results',
    'show', 'shown', 'present', 'presents', 'presented',
    'study', 'new', 'novel', 'use', 'used', 'also', 'work',
    'two', 'one', 'first', 'abstract', 'approach', 'approaches'
    'state', 'states', 'method', 'methods', 'model', 'models',
    'result', 'results', 'findings', 'conclusion', 'conclusions',
    'of','art','state-of-the-art','state-of-art',
]

def deduplicate_keywords(topic_words, top_n=5):
    keywords = [
        word.strip()
        for word, _ in topic_words
        if isinstance(word, str) and word.strip()
    ]
    if not keywords:
        return []
    selected_ngrams = [kw for kw in keywords if ' ' in kw]
    covered = set()
    for ngram in selected_ngrams:
        for part in ngram.split():
            covered.add(part)
    result = []
    for kw in keywords:
        if ' ' not in kw and kw in covered:
            continue
        result.append(kw)
        if len(result) >= top_n:
            break
    for kw in keywords:
        if len(result) >= top_n:
            break
        if kw not in result:
            result.append(kw)
    return result[:top_n]

def compute_coherence(topic_model, documents, valid_topics):
    tokenized_docs = [doc.lower().split() for doc in documents]
    dictionary = Dictionary(tokenized_docs)

    topic_words_list = []
    for _, topic_row in valid_topics.iterrows():
        topic_id = topic_row['Topic']
        words = topic_model.get_topic(topic_id)
        if not words:
            continue
        # Flatten n-grams to constituent words for coherence calculation
        flattened = []
        for word, _ in words[:10]:
            if not isinstance(word, str):
                continue
            flattened.extend(word.split())
        # Deduplicate while preserving order
        seen = set()
        unique_words = [w for w in flattened if not (w in seen or seen.add(w))]
        if unique_words:
            topic_words_list.append(unique_words[:10])

    if not topic_words_list:
        return 0.0

    coherence_model = CoherenceModel(
        topics=topic_words_list,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return coherence_model.get_coherence()

def build_filtered_vocab(documents, stop_words, min_df, ngram_range, max_features):
    pre_vec = CountVectorizer(
        stop_words=stop_words,
        min_df=min_df,
        ngram_range=ngram_range,
        max_features=max_features
    )
    pre_vec.fit(documents)

    return CountVectorizer(
        stop_words=stop_words,
        vocabulary=pre_vec.vocabulary_
    )

def split_and_clean(value):
    if pd.isna(value):
        return []
    items = str(value).split('; ')
    cleaned = []
    for item in items:
        c = clean_text(item)
        if c:
            cleaned.append(c)
    return cleaned

for _, row in macro_groups_df.iterrows():
    macro_id = row['Macro_Community']

    titles = split_and_clean(row['PaperTitles']) if 'PaperTitles' in row else []
    abstracts = split_and_clean(row['Abstracts']) if 'Abstracts' in row else []
    semantic_info = split_and_clean(row['SemanticInformation']) if 'SemanticInformation' in row else []

    max_len = max(len(titles), len(abstracts), len(semantic_info))
    documents = []
    for i in range(max_len):
        parts = []
        if i < len(titles):
            parts.append(titles[i])
        if i < len(abstracts):
            parts.append(abstracts[i])
        if i < len(semantic_info):
            parts.append(semantic_info[i])
        if parts:
            documents.append(". ".join(parts))

    print(f"Macro Community: {macro_id} - Documents: {len(documents)}")

    if len(documents) < 3:
        print("Skipping: Not enough documents for topic modeling")
        continue

    # Pre-filter vocabulary at document level, then wrap for BERTopic
    adaptive_min_df = max(2, int(len(documents) * 0.02))
    vectorizer_model = build_filtered_vocab(
        documents,
        stop_words=academic_stop_words,
        min_df=adaptive_min_df,
        ngram_range=(1, 3),
        max_features=2000
    )

    # Explicit UMAP for stable, coherent dimensionality reduction
    n_neighbors = min(max(8, int(0.1 * len(documents))), len(documents) - 1)
    n_components = max(3, min(5, len(documents)//3))
    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )

    # Explicit HDBSCAN for tighter clusters with less noise
    min_cluster = max(2, len(documents) // 15)
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster,
        min_samples=1,
        metric='euclidean',
        prediction_data=True
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        nr_topics='auto',
        top_n_words=10,
        calculate_probabilities=False,
        verbose=False
    )

    try:
        topics_result, _ = topic_model.fit_transform(documents)
        topic_info = topic_model.get_topic_info()

        valid_topics = topic_info[topic_info['Topic'] != -1]
        num_topics = len(valid_topics)

        print(f"  Topics discovered: {num_topics}")

        if num_topics > 0:
            # Compute coherence score
            coherence = compute_coherence(topic_model, documents, valid_topics)
            print(f"  Coherence (c_v): {coherence:.4f}")

            topic_details = []
            for _, topic_row in valid_topics.iterrows():
                topic_id = topic_row['Topic']
                topic_words = topic_model.get_topic(topic_id)
                keywords = deduplicate_keywords(topic_words, top_n=5)
                display_keywords = keywords if keywords else ["no_keywords"]
                score = topic_row['Count'] / len(documents)

                topic_details.append({
                    'topic_id': topic_id,
                    'keywords': display_keywords,
                    'count': topic_row['Count'],
                    'score': score
                })

                print(f"    Topic {topic_id}: {', '.join(display_keywords)} [Score: {score:.4f}]")

            macro_community_topics[int(macro_id)] = {
                'num_topics': num_topics,
                'topic_details': topic_details,
                'topic_model': topic_model,
                'num_documents': len(documents),
                'coherence': coherence,
                'documents': documents
            }
        else:
            print("  No distinct topics found")

    except Exception as e:
        print(f"  Error processing Macro Community {macro_id}: {e}")
        continue

In [ ]:
client    = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
LLM_MODEL = "gpt-5.2"         

def get_representative_docs(topic_model, documents: list[str],
                             topic_id: int, n: int = 15) -> list[str]:
    """Return up to *n* documents assigned to *topic_id*."""
    indices = [i for i, t in enumerate(topic_model.topics_) if t == topic_id]
    return [documents[i] for i in indices[:n]]

# ─────────────────────────────────────────────
# LLM call
# ─────────────────────────────────────────────
def generate_topic_label(keywords, representative_docs, model=LLM_MODEL):
    docs_text     = "\n".join(f"- {doc[:500]}" for doc in representative_docs[:100])
    keywords_text = ", ".join(keywords) if isinstance(keywords, list) else keywords

    prompt = f"""I have a topic from academic research that is described by the following keywords:
                {keywords_text}
                Representative documents in this topic:
                {docs_text}
                Based on the keywords and documents above, generate:
                1. A short, descriptive academic topic TITLE (3-10 words).
                2. A 2-4 sentence SUMMARY explaining the research theme.
                Format your response exactly as:
                Title: <title>
                Summary: <summary>"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are an expert academic research analyst. Generate concise, accurate topic labels."},
            {"role": "user",   "content": prompt}
        ],
        max_completion_tokens=500   
    )
    return response.choices[0].message.content.strip()


# ─────────────────────────────────────────────
# Response parser
# ─────────────────────────────────────────────
def parse_llm_response(response_text: str) -> tuple[str, str]:
    title = summary = ""
    for line in response_text.splitlines():
        line = line.strip()
        low = line.lower()
        if low.startswith("title:"):
            title = line[len("title:"):].strip()
        elif low.startswith("summary:"):
            summary = line[len("summary:"):].strip()
    return title, summary


# ─────────────────────────────────────────────
# Generate labels for all communities
# ─────────────────────────────────────────────
def generate_all_labels(macro_community_topics: dict,
                         model: str = LLM_MODEL,
                         rate_limit_sleep: float = 1.0) -> dict:
    llm_topic_labels: dict = {}

    for macro_id in sorted(macro_community_topics.keys()):
        data         = macro_community_topics[macro_id]
        topic_model  = data["topic_model"]
        documents    = data["documents"]
        topic_details = data["topic_details"]

        # Dominant topic = highest scoring topic in this community
        dominant         = max(topic_details, key=lambda td: td["score"])
        dominant_keywords = dominant["keywords"][:12]   # keep as list

        print(f"\nCommunity {macro_id}:")
        print("-" * 60)

        llm_topic_labels[macro_id] = {}
        rep_docs = get_representative_docs(
            topic_model, documents, dominant["topic_id"]
        )

        try:
            raw_response = generate_topic_label(
                dominant_keywords, rep_docs, model=model
            )
            title, summary = parse_llm_response(raw_response)

            llm_topic_labels[macro_id][dominant["topic_id"]] = {
                "title": title,
                "summary": summary,
            }

            print(f"Title:   {title}")
            print(f"Summary: {summary}")
            time.sleep(rate_limit_sleep)   # courtesy rate-limit pause

        except Exception as exc:
            print(f"  Topic Identification: Error – {exc}")
            llm_topic_labels[macro_id][dominant["topic_id"]] = {
                "title": "Label generation failed",
                "summary": "",
            }

    return llm_topic_labels

llm_topic_labels = generate_all_labels(
        macro_community_topics,
        model=LLM_MODEL,       
        rate_limit_sleep=1.0,
    )


In [ ]:
all_macro_ids = sorted(macro_groups_df['Macro_Community'].unique())

# Find the maximum number of topics across all communities
max_topics = max(
    (data['num_topics'] for data in macro_community_topics.values()),
    default=0
)

# Build the heatmap matrix (communities x topics) and collect keyword labels
heatmap_data = np.full((len(all_macro_ids), max_topics), np.nan)
keyword_labels = [['' for _ in range(max_topics)] for _ in range(len(all_macro_ids))]

for idx, mid in enumerate(all_macro_ids):
    if mid in macro_community_topics:
        for td in macro_community_topics[mid]['topic_details']:
            col = td['topic_id']
            if col < max_topics:
                heatmap_data[idx, col] = td['score']
                keyword_labels[idx][col] = ', '.join(td['keywords'][:3])

# Plot
fig, ax = plt.subplots(figsize=(max(12, max_topics * 3), max(6, len(all_macro_ids) * 0.6)))

# Use a masked array so NaN cells stay white
masked_data = np.ma.masked_invalid(heatmap_data)
cmap = plt.cm.YlOrRd
cmap.set_bad(color='whitesmoke')

im = ax.imshow(masked_data, cmap=cmap, aspect='auto', vmin=0, vmax=1)

# Axis labels
ax.set_xticks(np.arange(max_topics))
ax.set_xticklabels([f'Topic {i}' for i in range(max_topics)], fontsize=10, fontweight='bold')
ax.set_yticks(np.arange(len(all_macro_ids)))
ax.set_yticklabels([f'MC {mid}' for mid in all_macro_ids], fontsize=10)

# Annotate each cell with score and top keywords
for i in range(len(all_macro_ids)):
    for j in range(max_topics):
        if not np.isnan(heatmap_data[i, j]):
            score_text = f"{heatmap_data[i, j]:.0%}"
            kw_text = keyword_labels[i][j]
            color = 'white' if heatmap_data[i, j] > 0.5 else 'black'
            ax.text(j, i - 0.15, score_text, ha='center', va='center',
                    color=color, fontsize=8, fontweight='bold')
            ax.text(j, i + 0.15, kw_text, ha='center', va='center',
                    color=color, fontsize=6, style='italic')

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Topic Prevalence Score', rotation=270, labelpad=20, fontsize=11, fontweight='bold')

ax.set_title('Topic Prevalence Heatmap Across Macro Communities', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Summary Statistics Table
all_macro_ids_list = sorted(macro_groups_df['Macro_Community'].unique())
summary_data = []
for macro_id in all_macro_ids_list:
    if macro_id in macro_community_topics:
        data = macro_community_topics[macro_id]
        topic_details = data['topic_details']
        scores = [td['score'] for td in topic_details]

        # Dominant topic = the one with the highest score
        dominant = max(topic_details, key=lambda td: td['score'])
        dominant_keywords = ', '.join(dominant['keywords'][:12])

        summary_data.append({
            'Macro_Community': macro_id,
            'Num_Topics': data['num_topics'],
            'Num_Documents': data['num_documents'],
            'Coherence_cv': data.get('coherence', 0.0),
            'Dominant_Keywords': dominant_keywords,
            'Min_Score': np.min(scores),
            'Max_Score': np.max(scores),
            'Avg_Score': np.mean(scores),
        })
    else:
        row = macro_groups_df[macro_groups_df['Macro_Community'] == macro_id].iloc[0]
        num_docs = len(str(row['PaperTitles']).split('; ')) if pd.notna(row['PaperTitles']) else 0
        summary_data.append({
            'Macro_Community': macro_id,
            'Num_Topics': 0,
            'Num_Documents': num_docs,
            'Coherence_cv': np.nan,
            'Dominant_Keywords': 'N/A (failed)',
            'Min_Score': np.nan,
            'Max_Score': np.nan,
            'Avg_Score': np.nan,
        })

summary_df = pd.DataFrame(summary_data)

summary_csv_path = DIRECTORY / "summary_statistics_table1.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"Saved summary statistics to: {summary_csv_path}")

print("\n" + "=" * 130)
print("SUMMARY STATISTICS: Topic Analysis Across Macro Communities")
print("=" * 130 + "\n")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 60)

print(summary_df.to_string(index=False))
print("\n" + "=" * 130)

# Overall insights (only from successful communities)
successful = summary_df.dropna(subset=['Coherence_cv'])
total_topics = successful['Num_Topics'].sum()
total_docs = summary_df['Num_Documents'].sum()
avg_coherence = successful['Coherence_cv'].mean()


print(f"\nOVERALL INSIGHTS:")
print(f"  - Total Communities: {len(summary_df)} ({len(successful)} successful, {len(summary_df) - len(successful)} failed)")
print(f"  - Total Topics Discovered: {total_topics}")
print(f"  - Total Documents Analyzed: {total_docs}")
print(f"  - Average Topics per Community: {successful['Num_Topics'].mean():.1f}")
print(f"  - Average Coherence (c_v): {avg_coherence:.4f}")
print(f"  - Best Coherence: MC {successful.loc[successful['Coherence_cv'].idxmax(), 'Macro_Community']} ({successful['Coherence_cv'].max():.4f})")
print(f"  - Worst Coherence: MC {successful.loc[successful['Coherence_cv'].idxmin(), 'Macro_Community']} ({successful['Coherence_cv'].min():.4f})")
print(f"  - Community with Most Topics: MC {successful.loc[successful['Num_Topics'].idxmax(), 'Macro_Community']} ({successful['Num_Topics'].max()} topics)")
print(f"  - Community with Most Documents: MC {summary_df.loc[summary_df['Num_Documents'].idxmax(), 'Macro_Community']} ({summary_df['Num_Documents'].max()} docs)")
print("=" * 130)